# v0.3 · Notebook 06: Fine-tune mDeBERTa-v3

**LLM Security Gateway** · trains the gateway's second detection layer: a multilingual transformer that reads a message and outputs how likely it is to be a prompt injection.

**Why mDeBERTa-v3-base?** It's a strong, compact text-understanding model (MIT licence) pre-trained on 100 languages, so what it learns from English attacks partly transfers to Spanish. Plain DeBERTa-v3 is English-only, which is exactly why ProtectAI's detector failed on Spanish.

**What "fine-tuning" means here:** the model already understands language. We add a small classification layer on top and train the whole thing for a few passes over our `train` split, so it learns *our* task: injection or not.

**The targets** (from notebook 05, the best baseline on each column):

| | Best baseline | Target |
|---|---|---|
| Recall on human-translated Spanish | 77% (ProtectAI) | ≥ 80% |
| Recall on unseen attack styles | 48% (TF-IDF) | ≥ 70% |
| False positives on harmless emails | 9.6% (ProtectAI) | ≤ 5% |
| False positives on test | 3.8% (ProtectAI) | ≤ 2% |

Same rules as the baselines: threshold picked on `validation` (at most 1% of harmless messages blocked), test sets never used for any decision.

**Runtime: T4 GPU required** (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*). About 30–40 minutes. At the end the model is uploaded to your Hugging Face account as a **private** model, so a Colab disconnect can't lose it. Needs the `HF_TOKEN` secret (🔑 icon, *Acceso del cuaderno* on).

In [ ]:
!pip -q install datasets sentencepiece

In [ ]:
import os, sys, json, time, math, random, subprocess
import numpy as np
import pandas as pd

CFG = {
    "dataset": "edithngalame/prompt-injection-en-es",
    "revision": "v0.2.1",
    "base_model": "microsoft/mdeberta-v3-base",
    "hub_model": "edithngalame/mdeberta-v3-prompt-injection-en-es",
    "max_len": 512,          # tokens; longer texts are cut (we measure how often below)
    "batch_size": 8,
    "grad_accum": 2,         # 8 x 2 = effective batch of 16, fits comfortably in a T4's memory
    "lr": 2e-5,
    "epochs": 3,
    "warmup": 0.1,           # first 10% of steps: learning rate ramps up gently
    "weight_decay": 0.01,
    "fp16": True,            # half precision: about 2x faster on a T4. Set False if the loss shows nan
    "seed": 42,
    "target_fpr": 0.01,
    "out_dir": "/content/v06_mdeberta",
    "repo": "https://github.com/edithngalame/llm-security-gateway",
    "repo_dir": "/content/llm-security-gateway",
}
EVAL_SETS = ["test", "test_heldout", "test_native_es", "test_other_lang", "eval_handwritten"]
pd.set_option("display.max_colwidth", 140)
os.makedirs(CFG["out_dir"], exist_ok=True)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])

In [ ]:
import torch
torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "NONE: switch the runtime to T4 GPU before continuing")

In [ ]:
# The repo holds the baseline results we compare against
if not os.path.exists(CFG["repo_dir"]):
    subprocess.run(["git", "clone", "-q", "--depth", "1", CFG["repo"], CFG["repo_dir"]], check=True)
BASELINES = json.load(open(f'{CFG["repo_dir"]}/docs/results/baselines/results.json'))
assert BASELINES["dataset_revision"] == CFG["revision"], "baselines were measured on a different dataset version"
print("baselines:", ", ".join(BASELINES["detectors"]))

## 1. Load the dataset

In [ ]:
from datasets import load_dataset
dd = load_dataset(CFG["dataset"], revision=CFG["revision"])
data = {name: dd[name].to_pandas() for name in dd}
for name, d in data.items():
    print(f"{name:17s} {len(d):6,} rows  {int(d.label.sum()):6,} injections")

## 2. Scoring helpers

Identical to notebook 05, so the numbers are directly comparable.

In [ ]:
from sklearn.metrics import average_precision_score

def rates(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    tp = int(((y == 1) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    fp = int(((y == 0) & (pred == 1)).sum()); tn = int(((y == 0) & (pred == 0)).sum())
    return {"n": len(y), "attacks": tp + fn, "benign": fp + tn,
            "recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "fpr": fp / (fp + tn) if fp + tn else np.nan}

def slices(df):
    # Named groups of rows to report on (a row can be in several)
    is_email = df.text.str.contains("SUBJECT:", regex=False)
    return {
        "all": np.ones(len(df), bool),
        "lang=en": (df.lang == "en").values,
        "lang=es": (df.lang == "es").values,
        "source=user": (df.source == "user").values,
        "source=retrieved": (df.source == "retrieved").values,
        "hard benign": ((df.label == 0) & df.family.fillna("").str.startswith("hard_benign")).values,
        "harmless emails": ((df.label == 0) & is_email).values,
    }

def pick_threshold(scores, y, target_fpr):
    # Lowest threshold that blocks at most target_fpr of the harmless messages
    benign = np.sort(np.asarray(scores)[np.asarray(y) == 0])[::-1]
    k = int(np.floor(target_fpr * len(benign)))       # how many harmless ones may score above it
    return float(np.nextafter(benign[k], 1.0)) if k < len(benign) else 0.0

DETECTORS = {}   # name -> {"scores": {set: array}, "preds": {set: array}, "threshold", "ms_per_text"}

def register(name, scores, preds, threshold, ms_per_text, note=""):
    DETECTORS[name] = {"scores": scores, "preds": preds, "threshold": threshold,
                       "ms_per_text": ms_per_text, "note": note}
    rows = []
    for s in EVAL_SETS:
        df, p = data[s], preds[s]
        for sl, mask in slices(df).items():
            if mask.sum():
                rows.append({"eval_set": s, "slice": sl, **rates(df.label.values[mask], p[mask])})
    t = pd.DataFrame(rows)
    main = t[t.slice == "all"].set_index("eval_set")[["n", "attacks", "recall", "fpr", "precision"]]
    print(f"{name}: threshold {threshold if isinstance(threshold, str) else f'{threshold:.3f}'}, "
          f"{ms_per_text:.2f} ms per text on CPU")
    display(main.style.format({"recall": "{:.1%}", "fpr": "{:.1%}", "precision": "{:.1%}"}, na_rep="–"))

def cpu_latency(fn, texts, n=200):
    # Average time to score ONE message at a time, like the gateway does
    texts = list(texts)[:n]
    t0 = time.perf_counter()
    for t in texts:
        fn([t])
    return (time.perf_counter() - t0) * 1000 / len(texts)

In [ ]:
def pick(name, s, sl, metric):
    # s = one eval set, or "ALL" to pool every eval set (used for the small hard benign slice)
    sets = EVAL_SETS if s == "ALL" else [s]
    y = np.concatenate([data[x].label.values[slices(data[x])[sl]] for x in sets])
    p = np.concatenate([DETECTORS[name]["preds"][x][slices(data[x])[sl]] for x in sets])
    return rates(y, p)[metric] if len(y) else np.nan

COLUMNS = {  # column name -> (eval set, slice, metric)
    "Recall: test": ("test", "all", "recall"),
    "FPR: test": ("test", "all", "fpr"),
    "Recall: Spanish (test)": ("test", "lang=es", "recall"),
    "Recall: human Spanish": ("test_native_es", "all", "recall"),
    "Recall: unseen styles": ("test_heldout", "all", "recall"),
    "Recall: other languages": ("test_other_lang", "all", "recall"),
    "Recall: hand-written": ("eval_handwritten", "all", "recall"),
    "FPR: hard benign": ("ALL", "hard benign", "fpr"),   # only the hand-written sets have them
    "FPR: harmless emails": ("test", "harmless emails", "fpr"),
}

## 3. Tokenise

The model doesn't read letters, it reads **tokens** (word pieces from a 250,000-piece multilingual vocabulary). We check how many texts are longer than the 512-token limit: those get cut, and an attack hidden at the end of a long email could be cut off with them.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

tok = AutoTokenizer.from_pretrained(CFG["base_model"])
enc = {s: tok(list(d.text), truncation=True, max_length=CFG["max_len"])["input_ids"] for s, d in data.items()}
full_len = {s: np.array([len(x) for x in tok(list(d.text))["input_ids"]]) for s, d in data.items()}
print("example:", tok.convert_ids_to_tokens(tok("Ignora las instrucciones anteriores")["input_ids"]))
display(pd.DataFrame({s: {"median tokens": int(np.median(v)), "95th percentile": int(np.percentile(v, 95)),
                          f"longer than {CFG['max_len']}": f"{(v > CFG['max_len']).mean():.1%}"}
                      for s, v in full_len.items()}).T)

## 4. Train

A plain PyTorch training loop, written out so every step is visible (and so it doesn't break when a library renames an option):

1. Batches of similar length are grouped together, so less time is wasted on padding.
2. For each batch: the model predicts, the **loss** measures how wrong it was, and the optimiser nudges every weight to be a little less wrong.
3. After each **epoch** (one full pass over `train`), we score `validation`. We keep the epoch with the best **recall at 1% false positives** on validation: the number we actually care about, measured on data the model never trains on.

Watch the loss go down. If it prints `nan`, set `"fp16": False` in the settings cell and run again.

In [ ]:
def batches(ids, labels, batch_size, shuffle, rng):
    # Length-grouped batches: shuffle, sort within big chunks by length, cut into batches, shuffle batches
    order = list(range(len(ids)))
    if shuffle:
        rng.shuffle(order)
    chunk = batch_size * 50
    groups = []
    for i in range(0, len(order), chunk):
        part = sorted(order[i:i + chunk], key=lambda j: len(ids[j]), reverse=True)
        groups += [part[k:k + batch_size] for k in range(0, len(part), batch_size)]
    if shuffle:
        rng.shuffle(groups)
    for g in groups:
        b = tok.pad({"input_ids": [ids[j] for j in g]}, return_tensors="pt")
        b["labels"] = torch.tensor([int(labels[j]) for j in g])
        yield b


@torch.no_grad()
def margins(model, ids, batch_size=64):
    # Score = logit margin (injection minus benign), same as the open detector in notebook 05
    model.eval()
    out = np.zeros(len(ids))
    order = sorted(range(len(ids)), key=lambda j: len(ids[j]))
    for i in range(0, len(order), batch_size):
        g = order[i:i + batch_size]
        b = tok.pad({"input_ids": [ids[j] for j in g]}, return_tensors="pt").to(model.device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG["fp16"] and model.device.type == "cuda"):
            logits = model(**b).logits.float()
        out[g] = (logits[:, 1] - logits[:, 0]).cpu().numpy()
    return out


def recall_at_fpr(scores, y, fpr):
    t = pick_threshold(scores, y, fpr)
    return rates(y, (scores >= t).astype(int))["recall"], t

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["base_model"], num_labels=2, id2label={0: "benign", 1: "injection"}, label2id={"benign": 0, "injection": 1}
).float().to(device)   # .float(): newer library versions keep the checkpoint's half-precision weights, and
                       # mixed-precision training needs full-precision (fp32) weights to update
print("weights:", next(model.parameters()).dtype)

train_ids, train_y = enc["train"], data["train"].label.values
val_ids, val_y = enc["validation"], data["validation"].label.values
steps_per_epoch = math.ceil(len(train_ids) / CFG["batch_size"] / CFG["grad_accum"])
total_steps = steps_per_epoch * CFG["epochs"]

no_decay = ("bias", "LayerNorm.weight")
params = [{"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": CFG["weight_decay"]},
          {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0}]
optimizer = torch.optim.AdamW(params, lr=CFG["lr"])
scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG["warmup"] * total_steps), total_steps)
use_amp = CFG["fp16"] and device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp) if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") \
    else torch.cuda.amp.GradScaler(enabled=use_amp)

rng = random.Random(CFG["seed"])
log, best = [], {"recall": -1}
t_start = time.time()
for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    running, n_batches = 0.0, 0
    for i, b in enumerate(batches(train_ids, train_y, CFG["batch_size"], True, rng)):
        b = b.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = model(**b).loss / CFG["grad_accum"]
        if not torch.isfinite(loss):
            raise RuntimeError('loss is nan/inf: set "fp16": False in the settings cell and run again')
        scaler.scale(loss).backward()
        if (i + 1) % CFG["grad_accum"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        running += loss.item() * CFG["grad_accum"]; n_batches += 1
        if n_batches % 200 == 0:
            print(f"  epoch {epoch} · batch {n_batches:,} · loss {running / n_batches:.4f} · {(time.time() - t_start) / 60:.1f} min", flush=True)
    if n_batches % CFG["grad_accum"]:            # leftover gradients from the last few batches
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()

    val_scores = margins(model, val_ids)
    rec, thr = recall_at_fpr(val_scores, val_y, CFG["target_fpr"])
    pr_auc = average_precision_score(val_y, val_scores)
    log.append({"epoch": epoch, "train_loss": running / n_batches, "val_pr_auc": pr_auc,
                "val_recall_at_1pct_fpr": rec, "minutes": (time.time() - t_start) / 60})
    print(f"epoch {epoch}: train loss {running / n_batches:.4f} · validation PR-AUC {pr_auc:.4f} · "
          f"recall at 1% FPR {rec:.1%}", flush=True)
    if rec > best["recall"]:
        best = {"recall": rec, "epoch": epoch, "threshold": thr,
                "state": {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}}
        print(f"  ↑ best so far, kept", flush=True)

model.load_state_dict(best["state"])
THRESHOLD = best["threshold"]
print(f"\nusing epoch {best['epoch']} · threshold {THRESHOLD:.2f} (logit margin) · trained in {(time.time() - t_start) / 60:.0f} min")
display(pd.DataFrame(log).set_index("epoch").style.format({"train_loss": "{:.4f}", "val_pr_auc": "{:.4f}",
                                                           "val_recall_at_1pct_fpr": "{:.1%}", "minutes": "{:.0f}"}))

**Reading the table:** training loss should fall every epoch. If validation recall *falls* while training loss keeps falling, the model is starting to **memorise** the training set (overfitting): that's why we keep the best epoch, not the last one.

## 5. Evaluate on every test set

In [ ]:
scores = {s: margins(model, enc[s]) for s in EVAL_SETS}
preds = {s: (scores[s] >= THRESHOLD).astype(int) for s in EVAL_SETS}

model.to("cpu")                                  # the gateway runs on CPU, so time it there
def one(texts):
    b = tok(list(texts), truncation=True, max_length=CFG["max_len"], return_tensors="pt")
    with torch.no_grad():
        return model(**b).logits
ms = cpu_latency(one, data["test"].text, n=30)
model.to(device)

DETECTORS = {}
register("mDeBERTa-v3 (fine-tuned)", scores, preds, THRESHOLD, ms)

## 6. The scoreboard: fine-tuned model vs the baselines

Baseline rows are loaded from the repo (`docs/results/baselines/results.json`), measured by notebook 05 on the same dataset version.

In [ ]:
NAME = "mDeBERTa-v3 (fine-tuned)"
mine = {**{c: pick(NAME, *spec) for c, spec in COLUMNS.items()},
        "PR-AUC: test": average_precision_score(data["test"].label, DETECTORS[NAME]["scores"]["test"]),
        "CPU ms/text": DETECTORS[NAME]["ms_per_text"]}
board = pd.DataFrame({**{n: d["scoreboard"] for n, d in BASELINES["detectors"].items()}, NAME: mine}).T
board = board[list(mine)]
pct = [c for c in board.columns if c.startswith(("Recall", "FPR"))]
display(board.style.format({**{c: "{:.1%}" for c in pct}, "PR-AUC: test": "{:.3f}", "CPU ms/text": "{:.1f}"}, na_rep="–"))

TARGETS = {"Recall: human Spanish": (">=", 0.80), "Recall: unseen styles": (">=", 0.70),
           "FPR: harmless emails": ("<=", 0.05), "FPR: test": ("<=", 0.02)}
print("Targets:")
for col, (op, goal) in TARGETS.items():
    v = board.loc[NAME, col]
    ok = v >= goal if op == ">=" else v <= goal
    print(f"  {'MET    ' if ok else 'MISSED '} {col}: {v:.1%} (target {op} {goal:.0%})")

## 7. Chart

In [ ]:
import matplotlib.pyplot as plt

COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]   # fixed order: rules, TF-IDF, open detector, fine-tuned
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e6e5e0"

def grouped_bars(ax, cols, title, ymax=100):
    names = list(board.index)
    x = np.arange(len(cols)); w = 0.84 / len(names)
    for i, name in enumerate(names):
        vals = [board.loc[name, c] for c in cols]
        bars = ax.bar(x + (i - (len(names) - 1) / 2) * w, [0 if pd.isna(v) else v * 100 for v in vals],
                      w - 0.03, color=COLORS[i], label=name, zorder=2)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + ymax * 0.015, "–" if pd.isna(v) else f"{v:.0%}",
                    ha="center", va="bottom", fontsize=7.5, color=MUTED)
    ax.set_xticks(x, [c.split(": ", 1)[1] for c in cols], fontsize=9, color=INK)
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ticks = np.linspace(0, ymax, 5)
    ax.set_ylim(0, ymax * 1.1); ax.set_yticks(ticks, [f"{v:g}%" for v in ticks], fontsize=8, color=MUTED)
    ax.grid(axis="y", color=GRID, zorder=0); ax.tick_params(length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4.4), gridspec_kw={"width_ratios": [2.3, 1]})
grouped_bars(a1, ["Recall: test", "Recall: Spanish (test)", "Recall: human Spanish", "Recall: unseen styles",
                  "Recall: other languages", "Recall: hand-written"], "Attacks caught (higher is better)")
fpr_cols = ["FPR: test", "FPR: hard benign", "FPR: harmless emails"]
worst = np.nanmax(board[fpr_cols].values.astype(float)) * 100
grouped_bars(a2, fpr_cols, "Harmless messages blocked (lower is better)", ymax=max(10, 5 * np.ceil(worst / 5)))
fig.legend(*a1.get_legend_handles_labels(), frameon=False, fontsize=9, loc="upper left", ncol=len(board), bbox_to_anchor=(0.01, 1.07))
fig.tight_layout()
fig.savefig(f'{CFG["out_dir"]}/finetuned_vs_baselines.png', dpi=160, bbox_inches="tight")
plt.show()

## 8. Look at the mistakes

The attacks it still misses and the harmless messages it still blocks are the most useful output of this notebook: they tell you what data to add next.

In [ ]:
def show(df, cols=("text", "score", "lang", "source", "family"), n=12):
    display(df.loc[:, list(cols)].head(n).assign(text=lambda d: d.text.str.slice(0, 220)))

for s in ["test_native_es", "test_heldout", "eval_handwritten", "test"]:
    d = data[s].assign(score=scores[s])
    missed = d[(d.label == 1) & (preds[s] == 0)].sort_values("score")
    wrong = d[(d.label == 0) & (preds[s] == 1)].sort_values("score", ascending=False)
    print(f"\n{s}: missed {len(missed)} of {int(d.label.sum())} attacks · wrongly blocked {len(wrong)} of {int((d.label == 0).sum())} harmless")
    if len(missed): show(missed)
    if len(wrong): show(wrong)

## 9. Save: upload the model (private) and download the results

The model goes to **your** Hugging Face account as a *private* model: nobody else can see it until you decide. Notebook 07 will load it from there to export it for the gateway.

In [ ]:
from google.colab import userdata
token = userdata.get("HF_TOKEN")
model.config.update({"gateway_threshold_margin": float(THRESHOLD), "dataset": CFG["dataset"],
                     "dataset_revision": CFG["revision"], "best_epoch": best["epoch"]})
model.push_to_hub(CFG["hub_model"], private=True, token=token, commit_message="Fine-tuned on dataset v0.2.1")
tok.push_to_hub(CFG["hub_model"], private=True, token=token)
print(f"uploaded: https://huggingface.co/{CFG['hub_model']} (private)")

In [ ]:
import zipfile

by_slice = []
for s in EVAL_SETS:
    for sl, mask in slices(data[s]).items():
        if mask.sum():
            by_slice.append({"detector": NAME, "eval_set": s, "slice": sl, **rates(data[s].label.values[mask], preds[s][mask])})
pd.DataFrame(by_slice).to_csv(f'{CFG["out_dir"]}/results_by_slice.csv', index=False)
pd.concat([pd.DataFrame({"eval_set": s, "row": np.arange(len(data[s])), "label": data[s].label.values,
                         "score": scores[s]}) for s in EVAL_SETS]).to_parquet(f'{CFG["out_dir"]}/scores.parquet', index=False)

clean = lambda v: None if isinstance(v, float) and np.isnan(v) else round(float(v), 4)
results = {
    "release": "v0.3-finetuned",
    "model": CFG["hub_model"], "base_model": CFG["base_model"],
    "dataset": CFG["dataset"], "dataset_revision": CFG["revision"],
    "target_fpr_on_validation": CFG["target_fpr"],
    "settings": {k: CFG[k] for k in ("max_len", "batch_size", "grad_accum", "lr", "epochs", "warmup", "weight_decay", "fp16", "seed")},
    "best_epoch": best["epoch"], "threshold_margin": round(float(THRESHOLD), 4),
    "training_log": [{k: clean(v) if isinstance(v, float) else v for k, v in r.items()} for r in log],
    "scoreboard": {c: clean(v) for c, v in board.loc[NAME].items()},
}
json.dump(results, open(f'{CFG["out_dir"]}/results.json', "w"), indent=2, ensure_ascii=False)

ZIP = f'{"/content/" if os.path.exists("/content") else ""}v06_mdeberta_{CFG["revision"]}.zip'
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(CFG["out_dir"]):
        z.write(f'{CFG["out_dir"]}/{fn}', fn)
print(json.dumps(results["scoreboard"], indent=1))
print("\nsaved", ZIP)

In [ ]:
from google.colab import files
files.download(ZIP)

## 10. Questions to answer in your own words

1. Why does a *multilingual* pre-trained model help on the human-translated Spanish set, even though most of our Spanish training data is machine-translated?
2. Why do we pick the best epoch using validation and not test?
3. Compare the fine-tuned model with TF-IDF on "unseen styles". What does the difference suggest about what each one learned?
4. The model is about as slow on CPU as ProtectAI. Why might that be, and what could make it faster? (Notebook 07.)
5. Pick one attack it still misses. Why do you think it was missed, and what training data would fix it?